# 零、总述

本章我们来介绍多模态，如下图所示，如果一个语言模型可以像人类那样看一张图片，听一段语音等等，则此模型会被称为**多模态模型**。他不仅可以是**输入多模态**，也可以是**输出多模态**，也就是大模型可以生成图片，视频，语音等等内容

<center>
<img src="./resources/multimodal.png">
</center>

我们通常称**多模态(大)模型**为**Multimodal Large Language Model**, 而不是**Multimodal Large Model**，*这里要注意的是⚠️⚠️⚠️: 这里多加了 `language` 并没有翻译错*，而是因为 **<span style="color: red">Language（语言）绝不仅仅是一个孤立的词语。（英文原文：Language does not solely live in a vacuum）</span>**。

举个例子来说，你的*肢体语言、面部表情、说话的语调等等*都是交流的方式，它会让别人更加理解你在说什么。所以，**多模态模型**是 LLM 的一个扩展。

我们接下来讲一下**多模态模型的架构**，目前最广泛使用的架构为**Modality Encoder -> Projecter/Adapter -> LLM -> Output**，我们将这四个步骤分成了四层, 每一层的详细解释如下表所示

| **层级**      | **核心功能**                                  | **代表技术**                   | **输入 -> 输出**             |
|-------------|-------------------------------------------|----------------------------|--------------------------|
| 模态编码层       | 提取不同模态的信息                                 | ViT、CLIP、SigLIP、Whisper等   | 图像/音频/视频 -> 特征向量         |
| 模态对齐层       | 将不同模态的特征转换为 LLM 可有效利用的表示 | Linear、MLP Projecter、Q-Former等 | 模态特征向量 -> 对齐后的特征向量（多模态表示） |
| 多模态融合层 | 建立图像、文本等模态之间的关联，并进行理解和推理                  | Transformer、Self-Attention等 | 多模态表示 -> 隐藏状态     |
| 输出生成层       | 根据融合的结果生成目标模态                         | LM Head、Diffusion、Audio Decoder 等 | 隐藏状态 -> 文本/图像/音频/视频 |

① **模态编码层（Modality Encoding）**

**负责将原始信息转换成神经网络可以处理的特征表示**，可以使用 ViT, Video Encoder 等技术来编码多模态信息。CLIP 和 SigLIP 是一种预训练的方法或者是模型，Whisper 是语音识别的模型。

② **模态对齐层（Modality Alignment）**

它的核心目标是：**将视觉、音频等不同模态的特征转换为 LLM 能够有效处理和理解的表示。** 它主要有三个核心的功能：**维度对齐、语义适配和特征压缩**，这三个功能在模态对齐层不一定全部存在。其中，

- **维度对齐**

它的作用是**将视觉等模态特征映射到 LLM 所需的隐藏维度**，一般使用 Linear Projection 或 MLP Projector 来实现，Linear Projection 的底层是通过线性变换，而 MLP Projector 的底层是通过多层线性变换和非线性激活学习，更灵活的特征映射。

- **语义适配**

**它让映射后的特征能够被 LLM 有效利用，建立视觉信息与语言语义的关联**，一般使用 Q-former、经过多模态训练的 Projector 等来实现。

- **特征压缩**

**它主要是提取重要的模态信息，减少输入 LLM 的 token数量**，实现的技术一般是 Q-former 和 Resampler 等

③ **多模态融合层（Multimodal Fusion）**

模型对齐层解决的是 **不同模态的特征如何接入 LLM**，那么**多模态融合层**解决的就是**LLM 如何综合利用这些特征。**
这一层的核心作用是：**让不同模态的特征相互交互，建立跨模态关联，并结合上下文完成理解和推理**，它主要基于 Transformer 架构实现

④ **输出生成层（Output Generation）**

它负责将融合后的特征或条件信息转换为目标输出。常见的实现技术有 LM Head、Diffusion Model、Audio Decoder 和 Video Decoder/Generator 等。注意：**Diffusion Model** 不仅仅是一个简单的解码器，而是一套生成模型。


下面是详细的架构图

<center>
<img src="./resources/multimodal_architecture.png">
</center>

本章主要介绍视觉领域的一些技术，从 ViT 开始，随后讲解不同模态的编码器，接下来讲解如何让 LLM 可以处理多模态信息，最后介绍多模态生成的相关技术

# 一、视觉的 Transformer

我们可以将 Transformer 扩展到计算机视觉领域，这称之为 **ViT(Vision Transformer) 架构**，它在识别图像的任务上比之前的CNN 网络表现的性能更好。与 Transformer 类似，ViT 会将非结构化的数据（图片）转换为数据表示，以用于各种各样的任务当中，比如分类。如下图所示

<center>
<img src="./resources/Transformer_and_ViT.png">
</center>

ViT 使用了 Transformer 架构中比较重要的组件：**编码器**，它会将一段文本的输入转换为数字的表示，之后传到**解码器**中。然而，在编码器工作之前，文本的输入首先**需要分词（tokenize）**，即将一整个句子分成多个 token 的序列。因为图像不是一个词的序列，所以我们不能对其直接进行分词，但我们可以模拟分词这个思想，将图像分成每个“词”。

想象你有一张 512 x 512 像素的猫的图像，每一个单个的像素不代表什么，但是，你如果将这些像素进行分块（**patches**），它就能代表很多信息。ViT 使用了这种方式，**它把原始的图像转成了图像的分块（patches of images）**,也就是说，它将图片水平和竖直的切分成了块（patches）。如下图所示：

<center>
<img src="./resources/patched_image.png">
</center>

虽然，我们将其切成了块，这类似于文本中的词，但是，我们不能给这些块（patches）分配具体的 ID，因为在其他图片中，我们几乎很少能找见与之对应的块（patches）。那么，我们可以借助 embedding 模型来创建对应的数字表示，它可以作为 Transformer 模型的输入。这种方法（如下图所示）最有趣的地方在于，当这些嵌入传递到编码器之后，编码器会将它们视为文本的 token，从这一点来看，训练文本和图像没有任何区别。由于这种相似性，ViT 可以使得很多语言模型变得多模态，使用 ViT 最直接的方式就体现在训练嵌入模型的过程中。

<center>
<img src="./resources/ViT.png">
</center>

# 二、多模态编码与嵌入技术

在之前的章节中，我们使用了 Embedding 模型来捕捉文本表示的语义关系，比如论文或者文档的嵌入。在 LLM 的应用中，Embedding 是最重要的**引擎或者说是驱动者**，对于从大量的信息中提取某个信息，这种方式是最有效的，**现在我们可以将文本和视觉的表示都嵌入到同一个向量空间**中。如下图所示

<center>
<img src="./resources/multimodal_embedding_model.png">
</center>

这样的话，我们就可以基于文本来搜索图像了。例如：如果我们搜到了“小狗的图片”这段文字，那么它和哪个图像最接近呢？反过来，通过图片搜文字也是类似的操作。下图展示了嵌入空间

<center>
<img src="./resources/embedding_vector_space.png">
</center>

多模态的嵌入模型有很多，最有名的模型是**CLIP（Contrastive Language-Image Pre-training）**，与它类似的还有 *SigLIP/SigLIP2, DINOv2/DINOv3*，**CLIP、SigLIP/SigLIP2** 重点研究的图像与文本之间的语义对齐，**DINOv2/DINOv3** 重点研究如何从图像本身学习高质量的视觉特征，不依赖图文配对进行核心视觉预训练

## 2.1 CLIP(Contrastive Language-Image Pre-training)

### 2.1.1 基本介绍

CLIP 是一个嵌入模型，采用的是双编码器架构（Dual Encoder），它能计算图像和文本的嵌入，最终的嵌入向量依赖于相同的向量空间，它意味着图像可以和文本作比较，此模型可用于如下任务：

- Zero-shot classification
- Clustering
- Search
- Generation

### 2.1.2 CLIP 如何生成多模态的 Embedding

想象你有上百万张图片和对应的标题，那么这个数据集就能创建两个表示：图像和标题。

为了实现这个功能，CLIP 首先用文本编码器去嵌入文本并且用图像编码器去嵌入图像，这是第一步，如下图所示：

<center>
<img src="./resources/CLIP_First_Step.png">
</center>

接下来，CLIP 的第二步是去优化图文向量之间的相似度，我们使用图文相似度矩阵来展示，其训练目标是：**正确的图文对相似度尽可能高，错误的图文对相似度尽可能低**，如下图所示

<center>
<img src="./resources/CLIP_Second_Step.png">
</center>

在计算它们的相似度之后，模型的权重会被更新并且它会依赖于新一批的数据去更新表示，此方法叫做 Contrastive Learning，如下图所示

<center>
<img src="./resources/CLIP_Third_Step.png">
</center>

CLIP 使用 Softmax 作为它的损失函数，其公式如下：

$$
Loss_{CLIP} = \frac{1}{2} (Loss_{I \rightarrow T} + Loss_{I \rightarrow T})
$$

其中：

- $Loss_{I \rightarrow T}$ 代表 Image-To-Text 的损失，即沿着**每一行**计算 Softmax
- $Loss_{T \rightarrow I}$ 代表 Text-To-Image 的损失，即沿着**每一列**计算 Softmax

有一点需要说明的是，上述的两个方向是都需要计算的，因为**图像检索文本和文本检索图像是两个不同的条件分布**，即使它们都在同一个相似度矩阵中，但行方向和列方向的归一化分母不同，产生的损失和梯度也不同

### 2.1.3 OpenCLIP

我们使用开源的 CLIP 变体 OpenCLIP 来举例

In [ ]:
from urllib.request import urlopen
from PIL import Image

# 加载图像
puppy_path = "https://raw.githubusercontent.com/HandsOnLLM/Hands-On-Large-Language-Models/main/chapter09/images/puppy.png"
image = Image.open(urlopen(puppy_path)).convert("RGB")

caption = "a puppy playing in the snow"

因为我们有图像的标题，所以我们可以使用 OpenCLIP 来生成图像和文本的嵌入，我们加载三个模型

- 分词器：用来将文本输入分词
- 预处理器：用来预处理图像或调整图像大小
- **主模型**：将上述两个模型的输出转换为 Embedding 向量

In [ ]:
from transformers import CLIPTokenizer, CLIPProcessor, CLIPModel

model_id = "openai/clip-vit-base-patch32"

clip_tokenizer = CLIPTokenizer.from_pretrained(model_id)

clip_processor = CLIPProcessor.from_pretrained(model_id)

model = CLIPModel.from_pretrained(model_id)

在加载模型之后，我们直接预处理我们的输入，让我们将其分词并看看它会分成什么

In [ ]:
# 将输入分词
inputs = clip_tokenizer(caption, return_tensors="pt")
inputs

它输出了一个字典，里面包含输入的 ID，我们使用 `convert_ids_to_tokens` 来看一下每个 ID 都代表什么

In [ ]:
clip_tokenizer.convert_ids_to_tokens(inputs["input_ids"][0])

可以看到，文本的开始和结束表示我们应该从这两个地方将图像嵌入分割开。另外，你会注意到 `[CLS]` token 缺失了，实际上，在 CLIP 中，`[CLS]` token 被用于图像的表示。接下来，我们为输入创建 Embedding 向量

In [ ]:
# 创建文本嵌入
text_embedding = model.get_text_features(**inputs)
text_embedding.shape

我们接下来创建图像的嵌入，首先还是先预处理图像，之后再进行嵌入

In [ ]:
# 预处理图像
processed_image = clip_processor(
    text=None, images=image, return_tensors="pt"
)["pixel_values"]

processed_image.shape

原始的图像像素是 $512 \times 512$ 的，我们将它处理成了 $224 \times 224$ 的图像

我们来可视化我们的预处理结果

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt

img = processed_image.squeeze(0)
img = img.permute(*torch.arange(img.ndim - 1, -1, -1))
img = np.einsum("ijk->jik", img)

plt.imshow(img)
plt.axis("off")

随后，我们创建嵌入向量

In [ ]:
image_embedding = model.get_image_features(processed_image)
image_embedding.shape

可以看到图像和文本的嵌入具有相同的维度，那么我们就可以计算两个嵌入向量的相似度分数了

In [ ]:
# 归一化嵌入向量
text_embedding /= text_embedding.norm(dim=-1, keepdim=True)
image_embedding /= image_embedding.norm(dim=-1, keepdim=True)

# 计算相似度
text_embedding = text_embedding.detach().cpu().numpy()
image_embedding = image_embedding.detach().cpu().numpy()
score = np.dot(text_embedding, image_embedding.T)
score

我们得到了 0.33 的分数，但是我们并不清楚这个模型怎么去界定相似度的高低的，为此，我们会探索了更多的例子，如下图所示

<center>
<img src="./resources/similarity_matrix.png">
</center>

可以明显看到，0.33 这个分数确实比其他的分数要高（按行/列看）

## 2.2 SigLIP/SigLIP2

**SigLIP 和 SigLIP2 都是 Google 提出的视觉-语言编码模型（Vision-Language Encoder）**,主要用于学习图像与文本之间的语义关系，让图像和文本能够映射到可比较的特征空间。

两者的核心区别是：

- **SigLIP(2023)**：在 CLIP 双编码器架构的基础上，使用 Sigmoid Loss 替代 Softmax 对比损失，优化图文对齐训练
- **SigLIP(2025)**: 保留 SigLIP 的核心架构，增加了自监督学习、图像描述训练等技术，强化多语言理解、局部定位和密集视觉特征

### 2.2.1 SigLIP：基于 Sigmoid Loss 的图文对齐模型

SigLIP 全称是 Sigmoid Loss for Language-Image Pre-Training，它与 CLIP 的总体架构相似，采用的都是双编码器架构。它与 CLIP 的主要区别在于损失函数。

CLIP 使用的是 Softmax 对比损失，它会在一个 Batch 内构造图文相似度矩阵，通过 Softmax 让正确配对获得更高的相对概率。

SigLIP 使用的是 Sigmoid Loss, 它**将每个图文配对视为一个独立的二分类判断**

$$
P(匹配 | I, T) = \sigma(score(I,T))
$$

其中：

- $\sigma(·)$ 表示 Sigmoid 函数
- $score(I, T)$ 是图像和文本 Embedding 的相似度经过缩放、偏置处理后的得分

训练时，正样本的匹配概率趋近于 1， 负样本趋近于 0

### 2.2.2 SigLIP2：增强语义理解和空间感知

SigLIP2 的完整论文名称为：Multilingual Vision-Language Encoder with Improved Semantic Understanding, Localization, and Dense Features(2025 年 2 月发布)

SigLIP2 的主要升级是**通过多种训练目标，让视觉编码器不仅理解整张图片的语义，还能更好地保留局部区域的空间信息**

SigLIP2 的四项关键改进如下：

① **多语言理解（Multilingual Understanding）**

使用更丰富的多语言数据与训练策略，让图像能够更好地与中文、英文等不同语言的描述进行语义匹配，而不只是侧重英文

② **局部定位（Localization）**

通过加入带有位置监督的图像描述训练，让模型更好地理解物体和图像区域的关系。例如：模型不只知道图片中有一只猫，还能学习“左下角的猫”对应哪个区域

③ **密集视觉特征（Dense Features）**

强化各个图像 Patch 的特征表达，使它们包含更多局部语义和空间信息。这些特征更适合下游语义分割、深度估计等需要像素级或区域级理解的任务

④ **灵活分辨率（NaFlex）**

引入能够处理不同分辨率、保留原始宽高比的 NaFlex 版本，减少固定比例缩放带来的信息损失，对文档，长图等输入尤其有价值


下图展示了 SigLIP2 的架构，它分为三个部分

- **左侧：** EMA Image Encoder(Teacher), 通过自蒸馏增强视觉特征
- **中间：** Image Encoder + AR Decoder, 用于图像编码、图像描述和局部定位
- **右侧：** Text Encoder, 用于进行文本编码和语义对齐

其中，中间与右侧的图像、文本编码器构成了原始 SigLIP 的核心架构。SigLIP2 在此基础上增加了左侧的 EMA（Exponential Moving Average） 教师分支和上方的 AR Decoder 训练分支

<center>
<img src="./resources/SigLIP2_Architecture.jpg">
</center>

## 2.3 DINOv2/DINOv3

DINO 的名称来自 *Self-Distillation with No Labels，无标签自蒸馏*。 **无标签**说的是不需要依赖图像的文本编码，也可以让模型理解图片的语义，**这也是 DINO 系列的主要特点**。**自蒸馏** 的核心思想是让模型通过学习自身产生的监督信号，不断提高特征表达能力。

DINOv2/DINOv3 是 MetaAI 推出的视觉基础模型，主要通过自监督学习的方式从大量图像中提取通用视觉特征。

- **DINOv2(2023)**：重点是通过大规模自监督学习，获得稳定、通用的视觉特征，兼顾图像级语义和 Patch 级特征
- **DINOv3(2025)**：进一步扩大模型和数据规模，并通过 Gram Anchoring 改善长期训练中密集视觉特征退化的问题。

DINOv3 不仅能够理解图像的整体内容，还在物体区域、细节结构和空间关系等任务中表现突出

### 2.3.1 DINOv2

DINOv2 的完整论文名称为 *DINOv2: Learning Robust Visual Features without Supervision*，其核心思想是：**不需要人工告诉模型图片里面有什么，而是通过同一图片的不同视图之间的一致性学习，获得通用的视觉表示**

DINOv2 的视觉主干采用 ViT，训练时采用 Teacher-Student 自蒸馏结构, 下图展示了其训练的过程

<center>
<img src="./resources/DINOv2_Architecture.png">
</center>

整个训练过程可以理解为：

1. 输入一张图片，生成不同大小、不同增强方式的视图
2. Teacher 和 Student 分别处理这些视图，并使用 ViT 提取特征
3. 使用损失函数，让 Student 学习 Teacher 的语义表示
4. **Student 通过反向传播更新参数，Teacher 通过 EMA（Exponential Moving Average） 更新参数**

Teacher 不直接接受反向传播梯度，而是通过 Student 参数的移动平均进行更新，从而提供相对稳定的学习。


下面来讲一下 DINOv2 的三项核心技术：

① DINO Loss: 学习全局语义

例如，给模型输入同一只猫的不同裁剪图像，虽然裁剪的位置不同，但 Teacher 和 Student 应当提取具有一致语义的特征。它学习的是：**不同视图之间，哪些高层语义信息应该保持稳定**

② iBOT Loss: 学习局部特征

iBOT(Image BERT Pre-Training with Online Tokenizer)，它通过**掩码图像建模（Masked Image Modeling）** 进行训练，其过程为：

- Teacher 输入没有遮挡的图像
- Student 输入部分 Patch 被遮挡的图像
- Student 根据可见 Patch, 学习预测被遮挡位置的 Teacher 特征目标

这里主要预测的是视觉特征分布，不是直接重建图像像素

③ KoLeo Regularization: 避免特征过度集中

KoLeo 是一种用于鼓励特征分布更加均匀的正则化方法，它会避免不同图片得到的特征向量过度聚集在特征空间的某些区域，提升视觉表示的区分能力。

DINOv2 输出两类重要特征：

- **Global Feature(全局特征)**：通常使用 CLS Token 表示整张图像的语义，适合图像分类、图像相似度检索。
- **Patch Features(局部特征)**: 保留每个图像 Patch 对应的特征，适合目标区域分析、语义分割、深度估计等下游任务。


### 2.3.2 DINOv3

DINOv3 相比于 DINOv2 有以下的升级点：

- **训练数据**：由约 1.42 亿张扩展至 17 亿张图像
- **模型规模**：最大模型由约 1.1B 扩展至约 7B 参数
- **密集视觉特征**：引入 <span style="color: red">*Gram Anchoring*</span>，降低特征退化
- **位置编码**：使用 RoPE 等改进
- **模型部署**：提供 ViT 小模型及 ConvNeXt 版本
- **高分辨率处理**：通过后续适配增强高分辨率特征能力

**Gram Anchoring(Gram 锚定) 是 DINOv3 的核心创新**，在训练 DINOv2 的过程中发现：随着训练不断的进行，模型的图像分类能力可能继续提升，但 Patch 特征的局部一致性反而下降，导致下游的图像分割、定位等密集预测任务的表现变差。

如下图所示，图中的羊和花朵中的特征都随着训练集的增大都慢慢的变得语义不清晰，会有很多噪音

<center>
<img src="./resources/patch_level_feature.webp">
</center>

为了缓解这种退化，DINOv3 引入了 **Gran Anchoring**

具体原理如下：

假设一个图片被划分为 N 个 Patch，每个 Patch 都有一个 D 维的特征向量，将所有的 Patch 特征组织为 $F$，计算

$$
G = FF^T
$$

G 就是 Gram 矩阵，如果特征经过归一化，矩阵的每个元素就可以表示两个 Patch 特征之间的余弦相似度。

例如，如果你有 3 个 Patches: Tree, House, Road, 那么

$$
\mathrm{Gram\_matrix} =
\begin{bmatrix}
\mathrm{Tree}\cdot\mathrm{Tree} & \mathrm{Tree}\cdot\mathrm{House} & \mathrm{Tree}\cdot\mathrm{Road} \\
\mathrm{House}\cdot\mathrm{Tree} & \mathrm{House}\cdot\mathrm{House} & \mathrm{House}\cdot\mathrm{Road} \\
\mathrm{Road}\cdot\mathrm{Tree} & \mathrm{Road}\cdot\mathrm{House} & \mathrm{Road}\cdot\mathrm{Road}
\end{bmatrix}
$$

**Gram Anchoring 的训练流程为：**
1. 从训练较早的阶段保存一个具有良好局部特征的模型，作为**冻结的 Gram Teacher**
2. 当前 Student 提取图像 Patch 特征
3. 分别计算 Gram Teacher 和 Student 的 Patch 相似度矩阵
4. 通过损失函数约束两个 Gram 矩阵保持接近

这样，模型可以继续学习更丰富的全局语义，同时尽量保持原有的局部空间结构

这里要区分一点的是：<span style="color: orange">**Gram Teacher 是冻结的早期模型，而普通的 DINO Teacher 是通过 EMA 持续更新的教师模型**</span>


我们来直观的看一下，不同的模型对于**密集视觉特征**的提取能力，在大参数量的情况下，可以明显看到 **DINOv3** 比其他的模型表现的好得多

<center>
<img src="./resources/dense_features_comparison.webp">
</center>

# 三、模态对齐层：Bridging the Modality Gap

在多模态大模型的架构中，**模态对齐层（Modality Alignment Layer）** 的核心任务是：**将图像、音频、视频等不同模态的特征，转换成大语言模型能够有效利用的表示**，它可以按照实现机制分为三大类：


| 技术类别     | 代表技术                                                | 核心作用                   | 代表模型                |
|----------|-----------------------------------------------------|------------------------|---------------------|
| **投影映射**     | Linear Projection、MLP Projector                     | 特征维度映射、非线性特征转换         | LLaVA               |
| **查询对齐**     | Q-Former、Perceiver Resampler                        | 利用可学习 Query 提取、聚合和压缩特征 | BLIP-2, Flamingo    |
| **空间压缩与重采样** | Pixel Shuffle、Token Merging、Convolutional Projector | 减少视觉 Token，保留重要空间信息    | Qwen2.5-VL、Honeybee |


下面分别介绍这三类技术

## 3.1 投影映射

这类技术主要负责**特征映射**，而不是主动挑选、压缩视觉 Token

### 3.1.1 Linear Projection

线性映射，它是通过一个**线性函数**来使不同模态的特征维度对齐，其优点是可以保留原始 Token 的位置对应关系，缺点是仅能进行线性变换，难以表达更复杂的非线性映射。这个在 LLaVA 早期版本会被采用。

### 3.1.2 MLP Projector

**MLP Projector** 是 Linear Projection 的进一步扩展，它使用多层线性变换以及非线性激活函数，学习更复杂的视觉特征映射关系。

## 3.2 查询对齐

以 LLaVA 类模型为例，MLP Projector 可以将视觉特征映射到 LLM 的输入维度中，但是存在两个问题：

1. **视觉 Token 数量较多**

一张图片可能包含数百甚至数千个视觉 Token，如果全部传给 LLM，会增加序列长度和计算成本

2. **视觉特征不一定适合语言生成**

Vit 可能提取颜色、纹理、形状、局部位置等丰富视觉信息，但并不是所有信息都对当前语言任务有用。

例如，用户问：“这张图片里的人在做什么？”，那么模型就可能关注人物、姿势、手中的物品及其相互关系，而不一定需要完整保留每个背景区域的细节。

改进这两个问题的方法叫做**查询对齐**，即仅提取与用户输入问题相关的那部分视觉特征即可。

查询对齐的技术主要包括 Q-Former 和 Perceiver Resampler

### 3.2.1 Q-Former (in BLIP-2)

Q-Former 最具代表性的应用是 BLIP-2，它解决的问题是：**视觉编码器产生了大量特征，LLM 不一定需要处理全部特征，能够通过少量 Query 提取与语言任务相关的视觉信息？**

我们先讲 Q-Former 技术，之后介绍 BLIP-2 模型怎么训练的 Q-former

#### 3.2.1.1 Q-Former

Q-Former 会**让少量可学习的 Query 主动从视觉特征中聚合信息，形成紧凑的视觉表示，再交给语言模型**

Q-Former 的整体架构如图（简化版）,完整的见 [BLIP-2 论文](https://arxiv.org/abs/2301.12597)

<center>
<img src="./resources/Q_Former_Architecture.png">
</center>

从结构上看，Q-Former 可以理解为一个带有 Cross-Attention 的 Transformer 块，原始的论文中还有文本的 Transformer 块（图中未显示）。它有两条处理路径：**Image Transformer 和 Text Transformer**，两条路径会共享 Self-Attention 等 Transformer 参数，在不同训练任务中通过 Attention Mask 控制 Query 与文本之间的交互。

Image Transformer 的输入是 **Learnable Queries**, 它不是用户的问题，而是**一组可训练的参数向量**

接下来，我们看看 Q-Former 内部的 Attention 是如何工作的？

其核心计算机制是：

**Self-Attention + Cross-Attention + FFN**，其中。**Cross-Attention是完成视觉信息提取的关键**

**Self-Attention** 表示 Query 之间交换信息，它可以理解为：

- Query1 从其他 Query 获取上下文
- Query2 可以利用 Query1 的信息
- 其他 Query 也进行类似交互

这样，可以让 Query 不只是独立的处理信息，而是形成相关联的表示

所以，Self-Attention 不是直接读取图像，它处理的是 Query 自身的特征，**真正将外部视觉信息注入 Query 的，是 Cross-Attention**。

在 Cross-Attention 中：

| 参数 | 来源             | 作用        |
|----|----------------|-----------|
| Q  | 当前 Query 的隐藏状态 | 决定读取什么信息  |
| K  | ViT 的视觉特征      | 用于匹配视觉信息  |
| V  | ViT 的视觉特征      | 提供实际聚合的信息 |

那么，经过 Cross-Attention 公式计算后，其输出结果就会**对视觉信息进行加权聚合**。

FFN 仅仅是一个前馈神经网络，这里不再赘述

#### 3.2.1.2 BLIP-2

**BLIP-2：** *Bootstrapping Language-Image Pre-training for Unified Vision-Language Understanding and Generation 2*

我们从头开始创建一个多模态模型需要大量的算力，并且创建模型时，我们将会使用数十亿个图像、文本和图像文本对来训练，这种方式不太可行。我们可以转换另一个思路，无需从头构建，从大量的数据集中直接 Bootstrap(采样) 某些数据用来训练。BLIP-2 就使用了这种方法。

**BLIP-2**通过一个轻量级的查询 Transformer(Query Transformer, Q-Former)的方式来弥合视觉和语言之间的模态差距。Q-Former 采用两个阶段进行预训练：

- 第一阶段：利用**冻结**的*图像编码器*进行**视觉-语言表示学习（Vision-Language Representation）**
- 第二阶段：利用**冻结**的*语言模型*进行**视觉到语言的生成式学习（Vision-to-Language Generative Learning）**

两个阶段如下图所示

<center>
<img src="./resources/BLIP_2_stage.png">
</center>

我们在将其步骤细化一下

<center>
<img src="./resources/BLIP_2_procedure.png">
</center>



下面我们就详细介绍一下这两步

【第一步】

本阶段主要学习**视觉与语言之间的表示联系**，主要包含以下三类目标：

- **图像-文本对比学习（Image-text contrastive learning）**：此任务尝试对齐文本和图像的嵌入以最大化它们的互信息
- **图像文本匹配(Image-text Matching)**：一个分类任务，用来预测图像和文本是匹配（positive）或不匹配（negative）
- **基于图像生成文本(Image-grounded text generation)**


<center>
<img src="./resources/Q_Former_step_1.png">
</center>

【第二步】

由【第一步】生成的**可学习的嵌入向量**已经包含了视觉信息了，且维度已经统一，然后，这个**可学习的嵌入（Learnable Embedding）** 就可以被传递给 LLM。换句话说，**此嵌入被称作软视觉的提示词（soft visual prompts）, 它会在视觉表示方面对 LLM 条件化。**（英文原文：<span style="color: orange">In a way, these embeddings serve as soft visual prompts that *condition* the LLM *on* the visual representations that were extracted by the Q-Former.</span>）

另外，在 LLM 和 可学习的嵌入中会存在全连接层，那样可学习的嵌入向量就可以转换成 LLM 所期望的向量输入维度。

<center>
<img src="./resources/Q_Former_step_2.png">
</center>

#### 3.2.1.3 BLIP-2 示例

### 3.2.2 Perceiver Resampler(in Flamingo)

**感知重采样器**（Perceiver Resampler）是一种基于可学习潜在的查询和交叉注意力的视觉特征压缩与重采样技术。它的核心思想是：*使用固定数量的可学习 Latent Queries, 从大量、长度可变的视觉特征中提取信息，最终生成固定数量的视觉 Token，供后续 LLM 使用*，它最具代表性的应用是 Google DeepMind 提出的 Flamingo。在 Flamingo 中，Perceiver Resample 将视觉编码器输出的可变长度视觉特征**转换成固定的 64 个视觉 Token**，降低后续视觉与 Cross-Attention 的计算开销

## 3.3 空间压缩与重采样

**空间压缩与重采样（Spatial Compression & Sampling）**, 这类技术主要解决的问题是：**视觉编码器输出了大量的视觉 Token，如何在尽可能保留有效视觉信息的情况下，减少 Token 数量，降低后续 LLM 的计算和推理成本？**

- **空间压缩（Spatial Compression）:** 将相邻或相似的多个视觉特征聚合为更少的 Token，例如 Pooling、Patch Merging
- **空间采样（Spatial Sampling）：** 从空间特征中选取部分位置或区域，例如 Uniform Sampling、Top-K Sampling、Adaptive Token Selection

严格来说，二者并不完全互斥，某些基于注意力的采样方法，本身也会对视觉信息进行加权聚合.

主要代表技术如下表所示：

| 技术类别         | 代表技术                                         | 核心思想 |
|--------------|----------------------------------------------|------|
| 空间池化         | Average Pooling、Max Pooling、Adaptive Pooling | 聚合局部空间特征 |
| 空间降采样与重排     | Strided Convolution、Pixel Shuffle/Unshuffle  | 缩小特征图空间尺寸 |
| Patch 合并     | Patch Merging、Spatial Token Merger | 合并相邻视觉 Token |
| Token 选择与剪枝  | Uniform Sampling、Top-K、Token Pruning | 选择部分重要视觉 Token |
| Token 聚合与重采样 | Token Merging、TokenLearner | 将重要信息聚合成更少的 Token |

## 预处理多模态输入

## 使用示例：图像描述

## 使用示例：基于聊天的多模态提示词

# 四、多模态融合层

**多模态融合层（Multimodal Fusion Layer）** 的核心任务是：*让不同模态的信息发生交互，使 LLM 能够联合理解图像、文本、音频、视频等内容*。这一层涉及到的技术如下表所示

| 核心技术                                         | 主要作用 |
|----------------------------------------------|--|
| Token 拼接与交错（Token Concatenation/Interleaving） | 组织不同模态的 Token |
| Self-Attetion Fusion                         | 让多模态 Token 共同参与注意力计算 |
| Cross-Attention Fusion                       | 让一种模态选择性读取另一种模态的信息 |
| Gated Fusion                                 | 控制不同模态信息的注入强度 |
| Hierarchical Fusion                          | 在多个阶段逐步建立跨模态联系 |
| 时空感知融合（Spatiotemporal Fusion）            | 建立视觉空间、视频时间和文本之间的联系 |

# 五、多模态输出层

**多模态输出层** 的核心任务是：*将 LLM 或其他生成主干输出的隐藏状态，转换为文本、图像、音频、视频或结构化预测结果*。这里有一点需要说明的事：**对于具备高质量图像、视频生成能力的系统，此层更应该叫做<span style="color: red"> 多模态生成与输出层</span>，因为这层往往需要引入独立的<span style="color: red">生成主干(Generative Backbone)和模态解码器(Modality Decoder)</span>，而不仅仅是一个简单的输出头。**

截止到目前，多模态输出层有以下三条主线：

1. **文本生成**：Hidden States -> LM Head -> Token
2. **图像/视频生成**：条件特征 -> 自回归或连续生成模型 -> 视觉 Decoder
3. **音频生成**：条件特征 -> 音频 Token 或声学特征 -> Audio Decoder/Vocoder

## 5.1 文本生成

文本生成的整体流程如下图所示：

<center>
<img src="./resources/text_generation.png">
</center>

下面我只讲述 **LM Head, 语言模型输出头**，它是 LLM 的输出层，主要负责将 Transformer 产生的隐藏状态（Hidden States）映射到词表空间，从而预测下一个 Token，其本质上**通常是一个 Linear Layer**, 负责将隐藏状态映射到词表空间，LM Head 输出的是 Logits, 不是最终的文本，其输出维度由词表大小决定。

假设我们使用的模型隐藏维度为 4096，那么最后一个位置的隐藏状态可以表示为一个 4096 维的向量，但是**4096 维的隐藏状态并不能直接的表示应该输出哪个 Token**，假设我的词表中有 100,000 个 Token，那么，**LM Head 的工作就是将 4096 维向量<span style="color: red">映射</span>成 100,000 维向量**，我们称其为 **<span style="color: red">logits</span>**，每一个 logits 对应一个词

我们先来说**映射**，通常我们使用**线性映射**，其数学表达式为：

$$
z_t = h_t W_{LM} + b
$$

其中：

- $h_t$ 表示最后一个位置的隐藏状态
- $W_{LM}$ 表示 LM Head 的权重矩阵
- $b$ 表示偏置
- $z_t$ 表示输出的 logits


再来说 **logits**：**Logits 就是模型输出的原始预测分数**

## 5.2 图像/视频生成

**文本生成主要预测<span style="color: orange">离散 Token</span>，而图像和视频生成既可以预测<span style="color: orange">离散视觉 Token</span>，也可以在<span style="color: orange">连续潜空间</span>中生成视觉特征，最终在解码为像素**，图像、视频生成的核心技术可以从三个角度理解：

| 技术层级 | 核心技术                                   | 主要解决的问题         |
|------|----------------------------------------|-----------------|
| 生成方法 | Autoregressive、Diffusion、Flow Matching | 如何生成视觉内容？       |
| 生成网络 | Transformer、U-Net、DiT、MMDiT            | 使用什么神经网络生成？     |
| 视觉解码 | VQ Decoder、VAE Decoder、Video VAE       | 如何将模型表示转换为图像或视频？ |


这三个层级并不互斥，也可以是相辅相成的，例如，Flow Matching + DiT + VAE Decoder 就可以组成一个完整的图像生成系统。


接下来，从图像和视频生成这两个维度来介绍相关技术

### 5.2.1 图像生成

生成图像的技术路线主要有两条：**自回归图像生成** 和 **连续潜空间生成**

#### 5.2.1.1 自回归图像生成（Autoregressive Image Generation）

先来举个例子：

一开始,连续的图像向量会被 VQ 表示为

$$
I \rightarrow [V_1, V_2, ..., V_N]
$$

其中，$V_i$ 是离散视觉 token。随后，它会送到 Transformer（Autoregression） 中训练，这样就可以**学习**它们之间的条件概率关系

$$
P(V_1, ..., V_N | c) = \Pi_{i=1}^N P(V_i | V_{<i},c)
$$

其中，$c$ 是文本 prompt 等条件。

之后，就可以用已学习到的东西去逐步**预测**每一个视觉 Token，最后, 经过 **VQ Decoder** 就可以生成连续的向量，并恢复图像。

因此，*自回归图像生成* 的路线是
```text
文本 Prompt / 条件特征 -> Autoregressive Transformer -> VQ Decoder -> RGB 图像
```

这条路线与 LLM 的文本生成过程最为相似。核心思想是：**先将图像表示为一系列离散视觉 Token，在通过 Transformer 逐步预测这些 Token**。其中涉及了几项关键的技术

1. **Visual Tokenizer**(视觉分词器)

Visual Tokenizer 不像普通的分词器一样将其转换为 Token ID，而是将图像转换为离散编码，常用的技术是**VQ-VAE（Vector Quantized Variational Autoencoder）**, 主要过程是：原始图像 --> Visual Encoder --> 连续视觉特征 --> Vector Quantization(向量量化) --> 离散视觉 Token ID。

**向量量化的本质是：使用有限数量的代表性向量，近似表示连续空间中的任意向量。核心操作为：连续向量 --> 查找最近的码本向量 --> 离散索引**。

2. **VQ Decoder**

当视觉 Token 生成之后，我们需要通过对应码本来恢复量化的向量，此时就需要使用 VQ Decoder 来重建图像

#### 5.2.1.2 连续潜空间生成

*连续潜空间生成* 的路线是
```text
文本 Prompt / 条件特征 + 初始噪声 -> Diffusion / Flow Matching -> VAE Decoder -> RGB 图像
```

这里只讲两个技术：**Diffusion** 和 **Flow Matching**

1. Diffusion

2. Flow Matching

### 5.2.2 视频生成

## 5.3 音频生成